# W5C2: build a translator

English to Spanish on real Tatoeba sentences.

Most of this notebook is already written. Stages 4 and 7 are empty:
we write those two together.

## Stage 1. Load the pairs

Reads eng-spa.tsv and splits it three ways. accepted_es holds every Spanish
translation of a row, accepted_en every English one.

Checkpoint: 8761 train, 1094 dev, 1094 test.

In [ ]:
import pandas as pd
import torch
from torch import nn

df = pd.read_csv("exercise/data/eng-spa.tsv", sep="\t")
train = df[df["split"] == "train"]
dev = df[df["split"] == "dev"]
test = df[df["split"] == "test"]

print(len(train), "train,", len(dev), "dev,", len(test), "test")
print(df.head(3).to_string(index=False))

## Stage 2. Two vocabularies

Builds a word-to-id table per language. Ids 0, 1 and 2 are reserved for pad,
bos and eos. The Spanish table is built from accepted_es so every valid
translation is representable.

Checkpoint: 967 english, 1499 spanish.

In [ ]:
PAD, BOS, EOS = 0, 1, 2

def vocabulary(sentences):
    words = sorted({w for s in sentences for w in s.split()})
    return {w: i + 3 for i, w in enumerate(words)}

src_stoi = vocabulary(df["english"])
tgt_stoi = vocabulary(f for a in df["accepted_es"] for f in a.split("|"))
tgt_itos = {i: w for w, i in tgt_stoi.items()}

print(len(src_stoi) + 3, "english,", len(tgt_stoi) + 3, "spanish")

## Stage 3. Tensors

Encodes each sentence to ids and pads to a fixed width. The Spanish side is
wrapped in bos and eos first.

Checkpoint: [8761, 4] and [8761, 6].

In [ ]:
SRC_LEN = df["english"].str.split().str.len().max()
TGT_LEN = df["spanish"].str.split().str.len().max() + 2

def encode(sentence, stoi, width, wrap=False):
    ids = [stoi[w] for w in sentence.split()]
    if wrap:
        ids = [BOS] + ids + [EOS]
    return ids + [PAD] * (width - len(ids))

def tensors(frame):
    X = [encode(s, src_stoi, SRC_LEN) for s in frame["english"]]
    Y = [encode(s, tgt_stoi, TGT_LEN, wrap=True) for s in frame["spanish"]]
    return torch.tensor(X), torch.tensor(Y)

Xtr, Ytr = tensors(train)
Xte, Yte = tensors(test)

print(Xtr.shape, Ytr.shape)

## Stage 4. The model

Encoder LSTM, decoder LSTM started from the encoder's final state, linear layer
to the Spanish vocabulary.

## Stage 5. Teacher forcing

Slices the target twice, offset by one. y_in is what the decoder reads, y_out is
what each position predicts.

Checkpoint: both [8761, 5].

In [ ]:
y_in = Ytr[:, :-1]
y_out = Ytr[:, 1:]

print(Ytr.shape, "->", y_in.shape, y_out.shape)

## Stage 6. Train

Twenty passes over the training set with cross-entropy and ignore_index=PAD.

Checkpoint: final training loss 0.0158, about 30 seconds.

In [ ]:
from tqdm.auto import tqdm

torch.manual_seed(0)
model = Seq2Seq(len(src_stoi) + 3, len(tgt_stoi) + 3)
opt = torch.optim.Adam(model.parameters(), lr=0.002)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)

for epoch in tqdm(range(20), desc="training"):
    order = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 128):
        batch = order[i:i + 128]

        logits = model(Xtr[batch], Ytr[batch][:, :-1])
        loss = loss_fn(logits.reshape(-1, logits.size(-1)),
                       Ytr[batch][:, 1:].reshape(-1))

        opt.zero_grad()
        loss.backward()
        opt.step()

print(f"final training loss {loss.item():.4f}")

## Stage 7. Greedy decoding

Runs the decoder one step at a time, feeding its own output back in and carrying
the state forward.

## Stage 8. Sample translations

Prints five test sentences. A prediction counts if it matches any translation in
accepted_es.

Checkpoint: one of the five. "okay lets go" returns "pues vamonos", which is
correct Spanish but not the stored answer.

In [ ]:
def to_text(ids):
    words = []
    for i in ids.tolist():
        if i == EOS:
            break
        if i not in (PAD, BOS):
            words.append(tgt_itos[i])
    return " ".join(words)

for i in range(5):
    spanish = to_text(translate(model, Xte[i:i + 1])[0])
    mark = "ok " if spanish in test["accepted_es"].iloc[i].split("|") else "NO "
    print(f"{mark}{test['english'].iloc[i]:26} -> {spanish:28} (gold {test['spanish'].iloc[i]})")

## Stage 9. Free sentences

Translates four sentences, then one containing a word outside the vocabulary.

Checkpoint: the first four are correct; the fifth prints its unknown words.

In [ ]:
def translate_text(model, sentence):
    unknown = [w for w in sentence.split() if w not in src_stoi]
    if unknown:
        return f"<no id for: {' '.join(unknown)}>"
    x = torch.tensor([encode(sentence, src_stoi, SRC_LEN)])
    return to_text(translate(model, x)[0])

for sentence in ["a bird can fly",
                 "give me the key",
                 "he is my teacher",
                 "call your brother",
                 "i bought a laptop"]:
    print(f"{sentence:26} -> {translate_text(model, sentence)}")